In [ ]:
# distill_pytorch.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import tqdm

# --- Hyperparams ---
T = 4.0            # temperature
alpha = 0.7        # weight for soft-target loss
batch_size = 128
epochs = 30
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Data ---
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
])
train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4)

# --- Models: teacher (big) and student (small) ---
# For demo: teacher = pre-trained ResNet18 (pretend it's 'teacher'); student = small convnet
teacher = models.resnet18(pretrained=True)   # pretrained on ImageNet (not CIFAR) but OK for demo
# Replace final layer for CIFAR-10 (teacher must be trained ideally)
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
# NOTE: in practice load a real trained teacher checkpoint:
# teacher.load_state_dict(torch.load('teacher_ckpt.pth'))
teacher = teacher.to(device)
teacher.eval()  # important - teacher in eval mode

# Student: small CNN
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8*8*64, 256), nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

student = SmallCNN().to(device)

# --- Losses and optimizer ---
ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction='batchmean')  # expects input = log probs, target = probs
optimizer = torch.optim.SGD(student.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)

# --- Training loop (distillation) ---
for epoch in range(epochs):
    student.train()
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        # Teacher predictions (soft targets) -- compute without grads
        with torch.no_grad():
            teacher_logits = teacher(imgs)               # shape [B, C]
            teacher_probs_T = F.softmax(teacher_logits / T, dim=1)  # p^{(T)}

        # Student forward
        student_logits = student(imgs)                   # shape [B, C]
        student_log_probs_T = F.log_softmax(student_logits / T, dim=1)  # log q^{(T)}

        # Distillation (soft) loss: KL(teacher || student) at temperature T
        loss_kd = kl_loss(student_log_probs_T, teacher_probs_T) * (T * T)

        # Hard target loss
        loss_ce = ce_loss(student_logits, labels)

        # Combined loss
        loss = alpha * loss_kd + (1.0 - alpha) * loss_ce

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix({'loss': loss.item(), 'kd': loss_kd.item(), 'ce': loss_ce.item()})

    scheduler.step()

    # Quick eval
    student.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = student(imgs)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1} test acc: {acc:.4f}")

# Save student
torch.save(student.state_dict(), "student_distilled.pth")
